# wifi ensembles on the off-shell sig/bkg ratio — does an *asymptotic* uncertainty fix a *non-asymptotic* failure?

**Plan item 8e** (`docs/phase3-fixes.md`). **Ref. 17** = Benevedes & Thaler, *Frequentist Uncertainties on
Neural Density Ratios with wᵢfᵢ Ensembles*, [arXiv:2506.00113](https://arxiv.org/abs/2506.00113) — the method
the April meeting called the *"Wi-Fi ensemble"* (identified 2026-07-13; the name is a pun on weights **wᵢ**
× basis functions **fᵢ**, not a garbled transcription). Method write-up: foundations §15.1.

**Why this notebook exists.** wifi propagates the uncertainty of a *learned density ratio* into the parameter
of interest, analytically — no bootstrap loop. It is the natural network-side partner to Barlow–Beeston
(the template side, items 8a–8c). But **every one of its guarantees is asymptotic**: the sandwich covariance
is an M-estimator result (fixed M, N_fit → ∞) and the Gong–Samaniego propagation is a delta-method inflation
of σ²_MLE, which *presupposes* the Wald picture. This project exists because that picture fails at O(100)
events. So the sharp question (foundations §17.3):

> **Does an asymptotically-derived uncertainty inflation restore coverage in the regime defined by the
> failure of asymptotics?**

Both answers are informative. *Yes* → the missing ingredient was ratio-estimation noise, and the cure is cheap.
*No* → the overconfidence is **not** fitting-sample noise, and hypothesis 3 (sample-size blindness) survives a
serious challenge from the first method the field would reach for.

**What is being inferred.** Not μ. The full NSBI μ-scan needs `r_sbi/bkg`, which path-B training never learned
(it sat at the ln2 floor; path A is blocked on Tei). So we use the ratio that *does* train — `sig_over_bkg`
(val_loss 0.419, r rising 0.82 → 11.6 across m4ℓ quartiles) — and infer the **mixture fraction** κ of signal in
a sig+bkg mixture, exactly the task of the paper's own LHC demonstration (their quark fraction). Ref. 17 Eq. (27):

$$\log L(x|\kappa) = \log\big(\kappa\, r(x) + (1-\kappa)\big), \qquad r = p_\mathrm{sig}/p_\mathrm{bkg}.$$

This is a **methodology testbed, not a physics measurement**: κ is a mixture fraction, μ is a signal strength
with interference, and the two are not the same parameter. What transfers is the machinery and the answer to
the sharp question.

**The control the paper could not build.** Our events carry per-event matrix elements (`msq_sig_sm`,
`msq_bkg_sm`), so the **truth** density ratio is known analytically:
$r_\mathrm{true}(x) \propto \mathrm{msq_{sig}}(x)/\mathrm{msq_{bkg}}(x)$, normalised so $E_\mathrm{bkg}[r]=1$
(Ref. 17 Eq. 6). Benevedes & Thaler had no truth ratio for jets. We can therefore separate the two things wifi
treats differently: ratio **noise** (which it propagates) from ratio **bias** (which it explicitly does not).

**Engine:** `scripts/wifi_fit.py` (all compute). **Basis:** `scripts/wifi_train_basis.{sh,sub}` — M = 16
`sig_over_bkg` replicas, each on its own Poisson bootstrap of the training split
(`Process.resample`, the stub the sub-repo had already reserved *for wifi ensembles*).

In [ ]:
import os, sys, json, glob, importlib.util
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

ROOT = os.environ.get('SBI_ROOT', '/project/atlas/Users/mfernand/SBI_shortcomings')
SUB  = os.path.join(ROOT, 'asimov-aymptotics-check')
if SUB not in sys.path: sys.path.insert(0, SUB)

# the engine, imported (not reimplemented) so the notebook and the batch run share one code path
spec = importlib.util.spec_from_file_location('wifi_fit', os.path.join(ROOT, 'scripts/wifi_fit.py'))
wf = importlib.util.module_from_spec(spec); sys.modules['wifi_fit'] = wf; spec.loader.exec_module(wf)

RES = os.path.join(ROOT, 'output/wifi/results')
res = json.load(open(os.path.join(RES, 'wifi_results.json')))
cov = pd.read_csv(os.path.join(RES, 'coverage.csv'))
dg  = np.load(os.path.join(RES, 'diagnostics.npz'))

print('members trained :', len(res['checkpoints']))
print('ensemble sizes  :', list(res['ensembles']))
print('truth-ratio closure  E_bkg[r_true] = %.6f  (must be 1)' % res['closure_E_bkg_r_true'])
plt.rcParams.update({'figure.dpi': 110, 'font.size': 10})

## 1. Closure test of the engine — a case where the right answer is known exactly

Before trusting the machinery on networks, run it where the truth is analytic. Take $n=\mathcal N(1,1)$,
$d=\mathcal N(0,1)$, so $\log r(x) = x - \tfrac12$ **exactly**. Give the ensemble the basis $f = [x, 1]$: the
model is then *well-specified* (Ref. 17's key assumption) with a known answer, $w^* = (1, -\tfrac12)$.

Three things must hold: $\hat w \to w^*$; the pulls $(\hat w - w^*)/\sigma_w$ must be unit-normal (i.e. the
**sandwich covariance is honest**); and $C \sim O(N^{-1})$, so $\sigma_w$ must fall by $\sqrt{10}$ per decade.

In [ ]:
rng = np.random.default_rng(7)
W_TRUE = np.array([1.0, -0.5])
basis = lambda x: np.stack([x, np.ones_like(x)], axis=1)

rows = []
for N in [2_000, 20_000, 200_000]:
    fn, fd = basis(rng.normal(1, 1, N)), basis(rng.normal(0, 1, N))
    w = wf.fit_weights(fn, fd)
    C, _, _ = wf.weight_covariance(w, fn, fd)
    s = np.sqrt(np.diag(C))
    rows.append(dict(N=N, w0=w[0], w1=w[1], s0=s[0], s1=s[1],
                     pull0=(w[0]-1)/s[0], pull1=(w[1]+0.5)/s[1]))
clo = pd.DataFrame(rows)
clo['sigma_ratio_per_decade'] = clo.s0.shift(1) / clo.s0
print('w* = (1.0, -0.5)   sqrt(10) = %.2f\n' % np.sqrt(10))
display(clo.round(4))
print('\nCLOSURE: weights recovered, pulls O(1), sigma falls by ~sqrt(10)/decade => C ~ O(1/N) as Ref. 17 states.')

## 2. The basis ensemble: 16 bootstrapped `sig_over_bkg` replicas

Each member $f_i$ is a CARL classifier trained on its own **Poisson bootstrap** of the training split
(37% of events drop out per replica; the cross-section is renormalised). Initialisation is held *fixed*
across members on purpose: the diversity must come from the bootstrap — the thing whose uncertainty
wifi propagates — not from initialisation noise (which is what plain Deep Ensembles, Ref. 16, measure).

$f_i(x) \equiv \mathrm{logit}\,s_i(x)$ is member $i$'s estimate of $\log r(x)$; because CARL trains on
*balanced* classes, the logit is the log-ratio directly, with no prior correction.

In [ ]:
vl = [float(os.path.basename(c).split('val_loss=')[1].removesuffix('.ckpt')) for c in res['checkpoints']]
M_ref = int(dg['M_ref'])
fn_d, fd_d = dg['f_pool_n'], dg['f_pool_d']   # basis logits on the experiment pool

fig, ax = plt.subplots(1, 3, figsize=(14, 3.6))
ax[0].bar(range(1, len(vl)+1), vl, color='C0')
ax[0].axhline(np.log(2), color='r', ls='--', lw=1, label='ln2 = random floor')
ax[0].set_xlabel('member $i$'); ax[0].set_ylabel('best val_loss'); ax[0].set_ylim(min(vl)-0.02, np.log(2)+0.02)
ax[0].set_title('every basis net actually learned'); ax[0].legend()

for i in range(M_ref):
    ax[1].hist(fd_d[:, i], bins=60, range=(-4, 4), histtype='step', lw=0.7, alpha=0.7)
ax[1].set_xlabel(r'$f_i(x) = \log \tilde r_i(x)$  (bkg events)'); ax[1].set_ylabel('events')
ax[1].set_title(f'the {M_ref} basis functions (spread = bootstrap noise)')

spread = fd_d.std(axis=1)
ax[2].hist(spread, bins=60, color='C2')
ax[2].set_xlabel(r'member-to-member std of $f_i(x)$, per event'); ax[2].set_ylabel('events')
ax[2].set_title('per-event disagreement across the basis')
fig.tight_layout(); plt.show()

print('val_loss: min %.4f  max %.4f  (ln2 = %.4f)' % (min(vl), max(vl), np.log(2)))
print('median per-event spread of the basis: %.3f  (in log-ratio units)' % np.median(spread))

## 3. Fitting the weights: $\hat w$ vs the naive ensemble

The naive ensemble takes $w_i = 1/M$ (a plain average — Ref. 16). wifi instead **fits** the $w_i$ on a held-out
sample $D_\mathrm{fit}$ by minimising the symmetrised MLC loss (Ref. 17 Eq. 8), which is convex in $w$, and
reads off the sandwich covariance $C = V^{-1}UV^{-1}$ (Eq. 11). The paper's claim, which we can check directly:
**the naive uniform-weight ensemble undercovers and is biased; the fitted one does not.**

Note the three samples are strictly disjoint: `D_train` (trained the $f_i$) → `D_fit` (fits $\hat w$) →
`D_exp` (the experiments). Ours come from the 60/20/20 train/val/test split, with the test half split again.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
for M in sorted(res['ensembles'], key=int):
    e = res['ensembles'][M]
    w, s = np.array(e['w_hat']), np.array(e['sigma_w'])
    ax[0].errorbar(np.arange(1, int(M)+1), w, yerr=s, marker='o', ms=3, lw=1, capsize=2, label=f'M={M}')
    ax[0].axhline(1/int(M), color='grey', ls=':', lw=0.8)
ax[0].set_xlabel('member $i$'); ax[0].set_ylabel(r'$\hat w_i \pm \sigma_i$')
ax[0].set_title(r'fitted weights (dotted = naive $1/M$)'); ax[0].legend(fontsize=8)

Ms = sorted(map(int, res['ensembles']))
ax[1].plot(Ms, [res['ensembles'][str(M)]['loss_naive'] for M in Ms], 'o-', label='naive ($w_i=1/M$)')
ax[1].plot(Ms, [res['ensembles'][str(M)]['loss_wifi']  for M in Ms], 's-', label=r'wifi (fitted $\hat w$)')
ax[1].set_xlabel('M'); ax[1].set_ylabel('symmetrised MLC loss on $D_{fit}$')
ax[1].set_title('fitting the weights improves the ratio'); ax[1].legend()
fig.tight_layout(); plt.show()

for M in Ms:
    e = res['ensembles'][str(M)]
    print(f"M={M:2d}  sum(w)={np.sum(e['w_hat']):.3f}   loss {e['loss_naive']:.5f} -> {e['loss_wifi']:.5f}")

## 4. Learned ratio vs **truth** ratio — noise, or bias?

This is the control the paper could not run. wifi's covariance is an honest account of the *noise* in the
learned ratio. It says nothing about *bias* — a systematically wrong ratio. Having $r_\mathrm{true}$ from the
matrix elements, we can see which we have.

In [ ]:
lr_true = np.log(np.maximum(np.concatenate([dg['r_true_n'], dg['r_true_d']]), 1e-12))
lr_wifi = np.concatenate([dg['logr_wifi_n'], dg['logr_wifi_d']])
lr_naiv = np.concatenate([dg['logr_naive_n'], dg['logr_naive_d']])
sig_r   = np.sqrt(np.einsum('ij,jk,ik->i', np.vstack([dg['f_pool_n'], dg['f_pool_d']]),
                            dg['C_ref'], np.vstack([dg['f_pool_n'], dg['f_pool_d']])))  # Eq. (20)

fig, ax = plt.subplots(1, 3, figsize=(15, 3.8))
h = ax[0].hist2d(lr_true, lr_wifi, bins=80, range=[[-3, 4], [-3, 4]], cmin=1, cmap='viridis')
ax[0].plot([-3, 4], [-3, 4], 'r--', lw=1)
ax[0].set_xlabel(r'$\log r_{\rm true}$'); ax[0].set_ylabel(r'$\log \tilde r_{\rm wifi}$')
ax[0].set_title('learned vs truth log-ratio')

bins = np.linspace(-3, 4, 25); ctr = 0.5*(bins[1:]+bins[:-1])
idx = np.digitize(lr_true, bins) - 1
for lab, y, c in [('wifi', lr_wifi, 'C0'), ('naive', lr_naiv, 'C1')]:
    m = np.array([np.mean(y[idx == k] - lr_true[idx == k]) if (idx == k).sum() > 20 else np.nan
                  for k in range(len(ctr))])
    ax[1].plot(ctr, m, 'o-', ms=3, color=c, label=lab)
ax[1].axhline(0, color='r', ls='--', lw=1)
ax[1].set_xlabel(r'$\log r_{\rm true}$'); ax[1].set_ylabel(r'bias  $\langle \log\tilde r - \log r_{\rm true}\rangle$')
ax[1].set_title('BIAS of the learned ratio'); ax[1].legend()

ax[2].hist(sig_r, bins=60, color='C3')
ax[2].set_xlabel(r'wifi noise on $\log\tilde r$:  $\sigma(x)=\sqrt{f_i C_{ij} f_j}$  (Eq. 20)')
ax[2].set_ylabel('events'); ax[2].set_title('NOISE that wifi propagates')
fig.tight_layout(); plt.show()

rms_bias = np.sqrt(np.mean((lr_wifi - lr_true)**2))
print('RMS(log r_wifi - log r_true) = %.3f   [BIAS+noise, in log-ratio units]' % rms_bias)
print('median wifi sigma(log r)     = %.3f   [NOISE only, what Eq. (20) accounts for]' % np.median(sig_r))
print('\nIf the first number is much larger than the second, the learned ratio is dominated by MISMODELLING,')
print('which Ref. 17 explicitly does NOT cover -- and no amount of weight-covariance can repair it.')

## 5. The headline: coverage vs $N_\mathrm{mix}$ — into the non-asymptotic regime

68% Wald intervals, as in the paper. Four estimators of $r$, all inferring the same κ on the same toys:

| | ratio | interval |
|---|---|---|
| **truth** | exact $r_\mathrm{true}$ from matrix elements | $\sigma_\mathrm{MLE}$ |
| **naive** | uniform-weight ensemble (Ref. 16) | $\sigma_\mathrm{MLE}$ |
| **wifi (MLE)** | fitted $\hat w$ | $\sigma_\mathrm{MLE}$ — ignores ratio noise |
| **wifi (GS)** | fitted $\hat w$ | $\sigma_\mathrm{GS}$ — Gong–Samaniego, Eq. (22) |

The **truth** row is the reference: it isolates what is due to the *finite dataset alone*, with a perfect ratio.
Any gap between truth and wifi is the price of *learning* the ratio — and the wifi(MLE) → wifi(GS) step is
precisely the part of that price Ref. 17 claims to pay back.

In [ ]:
M_show = max(map(int, res['ensembles']))
kappas = sorted(cov.kappa_true.unique())
fig, axes = plt.subplots(1, len(kappas), figsize=(4.2*len(kappas), 3.8), sharey=True)

for ax, k in zip(np.atleast_1d(axes), kappas):
    d = cov[(cov.kappa_true == k) & (cov.M == M_show)].sort_values('n_mix')
    t, nv, wi = d[d.method=='truth'], d[d.method=='naive'], d[d.method=='wifi']
    ax.errorbar(t.n_mix,  t.coverage_mle,  yerr=t.coverage_err,  marker='o', color='k',  lw=1.2, label='truth ratio (oracle)')
    ax.errorbar(nv.n_mix, nv.coverage_mle, yerr=nv.coverage_err, marker='v', color='C1', lw=1.2, label='naive ensemble')
    ax.errorbar(wi.n_mix, wi.coverage_mle, yerr=wi.coverage_err, marker='s', color='C3', lw=1.2, label=r'wifi, $\sigma_{MLE}$')
    ax.errorbar(wi.n_mix, wi.coverage_gs,  yerr=wi.coverage_err, marker='D', color='C0', lw=1.6, label=r'wifi, $\sigma_{GS}$ (Eq. 22)')
    ax.axhline(0.683, color='g', ls='--', lw=1, label='nominal 68.3%')
    ax.set_xscale('log'); ax.set_xlabel(r'$N_{\rm mix}$  (events in the experiment)')
    ax.set_title(fr'$\kappa_{{\rm true}} = {k}$')
np.atleast_1d(axes)[0].set_ylabel('coverage of the 68% interval')
np.atleast_1d(axes)[0].legend(fontsize=7.5, loc='lower right')
fig.suptitle(f'Coverage vs experiment size  (M={M_show} basis networks)  —  left edge = the non-asymptotic regime', y=1.02)
fig.tight_layout(); plt.show()

display(cov[(cov.M == M_show)].pivot_table(index=['kappa_true','n_mix'], columns='method',
            values=['coverage_mle','coverage_gs']).round(3))

## 6. How much does Gong–Samaniego actually inflate, and does M help?

$\sigma_\mathrm{GS}/\sigma_\mathrm{MLE} = \sqrt{1 + \sigma^2_\mathrm{MLE} A_i C_{ij} A_j}$. Since
$C \sim O(1/N_\mathrm{fit})$ is *fixed* once the weights are fitted, while $\sigma^2_\mathrm{MLE} \sim 1/N_\mathrm{mix}$
grows as the experiment shrinks, the inflation is **largest exactly where we need it** — small $N_\mathrm{mix}$.
Whether "largest" is "large enough" is the empirical question.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
w = cov[cov.method == 'wifi']
for k in kappas:
    d = w[(w.kappa_true == k) & (w.M == M_show)].sort_values('n_mix')
    ax[0].plot(d.n_mix, d.inflation_mean, 'o-', ms=4, label=fr'$\kappa={k}$')
ax[0].axhline(1, color='grey', ls=':', lw=1)
ax[0].set_xscale('log'); ax[0].set_xlabel(r'$N_{\rm mix}$')
ax[0].set_ylabel(r'$\sigma_{GS}/\sigma_{MLE}$'); ax[0].set_title('the Gong-Samaniego inflation'); ax[0].legend(fontsize=8)

kk = kappas[0]
for M in sorted(map(int, res['ensembles'])):
    d = cov[(cov.method=='wifi') & (cov.kappa_true==kk) & (cov.M==M)].sort_values('n_mix')
    ax[1].errorbar(d.n_mix, d.coverage_gs, yerr=d.coverage_err, marker='o', ms=4, lw=1, label=f'M={M}')
ax[1].axhline(0.683, color='g', ls='--', lw=1)
ax[1].set_xscale('log'); ax[1].set_xlabel(r'$N_{\rm mix}$'); ax[1].set_ylabel(r'coverage, $\sigma_{GS}$')
ax[1].set_title(fr'does a bigger basis help?  ($\kappa={kk}$)'); ax[1].legend(fontsize=8)
fig.tight_layout(); plt.show()

print('Fraction of fits pinned at the kappa boundary (0 or 1) -- the non-asymptotic tell:')
display(cov[(cov.M==M_show) & (cov.method=='wifi')].pivot_table(
    index='n_mix', columns='kappa_true', values='frac_at_boundary').round(3))

## 7. Verdict

*(filled in from the numbers above — see `docs/phase3-fixes.md` item 8e for the write-up.)*